In [ ]:
import anndata as ad
import os
import re
import numpy as np
import scanpy as sc
import pandas as pd

In [ ]:
# Load single-nucleus nuclear protein intensity from one brain section
folder = "cell_measurement/ave_int/"      # <- your folder with outputs from Step7 

# collect all h5ad file paths
files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5ad")]

# read all into a list
adatas = [sc.read_h5ad(f) for f in files]

# concatenate into one AnnData
adata = ad.concat(adatas, join="outer", label="batch", keys=[os.path.basename(f) for f in files])

print(adata)

In [ ]:
# Load center-of-mass of Cellpose masks for spatial positions

spatial_df = pd.read_csv("input/center_of_mass_results_brain1.csv")    # <- output from Step_4_2
spatial_df['Label'] = spatial_df['Label'].astype(str)

spatial_df = spatial_df.set_index("Label")
spatial_df = spatial_df.loc[adata.obs_names]

spatial_ar = spatial_df[["X", "Y", "Z"]].to_numpy()
adata.obsm['spatial3d']=spatial_ar
adata.obsm['spatial']=spatial_ar[:,:-1]
adata.obs['z_layer']=spatial_ar[:,-1]

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

In [ ]:
adata.write_h5ad("output/adata_18_nuclear_46_prot_brain1.h5ad")   # same file provided in input/